In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
city_df = pd.read_csv('city_day.csv')


# Drop rows with missing target (AQI)
city_df = city_df.dropna(subset=["AQI"])

# Drop non-useful columns
city_df = city_df.drop(columns=["City", "Date", "AQI_Bucket"], errors="ignore")


# Separate features and target
X = city_df.drop("AQI", axis=1)
y = city_df["AQI"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
param_grid = {
    "max_depth":         [3, 5, 8, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 2, 5],
    "criterion":         ["squared_error", "absolute_error"],
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_

In [ ]:
dt_model = DecisionTreeRegressor(**best_params, random_state=42)
dt_model.fit(X_train, y_train)

In [ ]:
cv_r2   = cross_val_score(dt_model, X_train, y_train, cv=5, scoring="r2")
cv_rmse = cross_val_score(dt_model, X_train, y_train, cv=5,
                           scoring="neg_root_mean_squared_error")

In [ ]:
y_pred = dt_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"\n  MAE  (Mean Absolute Error):          {mae:.4f}")
print(f"  RMSE (Root Mean Squared Error):      {rmse:.4f}")
print(f"  R²   (Coefficient of Determination): {r2:.4f}")